In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [6]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.5803449153900146
epoch:  1, loss: 0.49631181359291077
epoch:  2, loss: 0.43167030811309814
epoch:  3, loss: 0.3420950770378113
epoch:  4, loss: 0.27060166001319885
epoch:  5, loss: 0.221078023314476
epoch:  6, loss: 0.17770938575267792
epoch:  7, loss: 0.1448339968919754
epoch:  8, loss: 0.11733856797218323
epoch:  9, loss: 0.081893689930439
epoch:  10, loss: 0.06967871636152267
epoch:  11, loss: 0.059039343148469925
epoch:  12, loss: 0.0218697227537632
epoch:  13, loss: 0.009698051027953625
epoch:  14, loss: 0.0071803126484155655
epoch:  15, loss: 0.005783421453088522
epoch:  16, loss: 0.004912945907562971
epoch:  17, loss: 0.004497092682868242
epoch:  18, loss: 0.003997999243438244
epoch:  19, loss: 0.003941430244594812
epoch:  20, loss: 0.003522473154589534
epoch:  21, loss: 0.003246048931032419
epoch:  22, loss: 0.0032161532435566187
epoch:  23, loss: 0.003006331855431199
epoch:  24, loss: 0.0028622793033719063
epoch:  25, loss: 0.0027136944700032473
epoch:  26, 

In [7]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9572124481201172
Test metrics:  R2 = 0.6942067742347717


In [8]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3006219267845154
epoch:  1, loss: 0.2683723270893097
epoch:  2, loss: 0.21636778116226196
epoch:  3, loss: 0.13754171133041382
epoch:  4, loss: 0.02767718769609928
epoch:  5, loss: 0.016494428738951683
epoch:  6, loss: 0.012940051034092903
epoch:  7, loss: 0.0094080101698637
epoch:  8, loss: 0.007435651961714029
epoch:  9, loss: 0.006313237827271223
epoch:  10, loss: 0.005804806016385555
epoch:  11, loss: 0.005189782474189997
epoch:  12, loss: 0.004697474185377359
epoch:  13, loss: 0.004369888920336962
epoch:  14, loss: 0.004105254542082548
epoch:  15, loss: 0.003890911815688014
epoch:  16, loss: 0.0037262337282299995
epoch:  17, loss: 0.0036208152305334806
epoch:  18, loss: 0.00351326959207654
epoch:  19, loss: 0.0033269114792346954
epoch:  20, loss: 0.003218753496184945
epoch:  21, loss: 0.0031219886150211096
epoch:  22, loss: 0.0030254023149609566
epoch:  23, loss: 0.0029635506216436625
epoch:  24, loss: 0.0028993585146963596
epoch:  25, loss: 0.002847619587555527

In [9]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9571130275726318
Test metrics:  R2 = 0.6869595050811768
